In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score as sklearn_silhouette_score
from customer_vectorization_functions import function1
from segmentation_functions import my_adjusted_kmeans, merge_kmeans, hierarchical_stratification

In [ ]:
#ucčitavanje podataka
df = pd.read_csv(
    r"C:\Users\lovro\Desktop\hackatoni\LUMEN_DS.csv",
    sep="|",
    quotechar='"',
    encoding="utf-16",
)

In [ ]:
#ovdje gledamo koliko svaki covjek/kompanija je napravio narudžbi i po toj mjeri micemo outliere, njih je 1%, njih posebno hendlat

n_orders_per_customer = df.groupby('CustomerID').size().reset_index(name='n_orders')
high_volume_threshold = n_orders_per_customer['n_orders'].quantile(0.99)  # Top 1%
high_volume_customers = n_orders_per_customer[
    n_orders_per_customer['n_orders'] > high_volume_threshold
]['CustomerID'].tolist()

print(f"High-volume customers (>{high_volume_threshold:.0f} orders): {len(high_volume_customers)} ({len(high_volume_customers)/len(n_orders_per_customer)*100:.1f}%)")

# Separate them
df_high_volume = df[df['CustomerID'].isin(high_volume_customers)]
df_regular = df[~df['CustomerID'].isin(high_volume_customers)]

In [ ]:
data_points = function1(df_regular)       

In [ ]:
import numpy as np

def metrics(data_points, labels):
    k = labels.max() + 1
    
    # Compute centroids
    centroids = np.vstack([
        data_points[labels == i].mean(axis=0)
        for i in range(k)
    ])
    
    # Silhouette
    silhouette = sklearn_silhouette_score(data_points, labels)
    
    # Mean distance to assigned centroid
    mean_centroid_distance = np.mean(
        np.linalg.norm(data_points - centroids[labels], axis=1)
    )
    
    return silhouette, mean_centroid_distance

In [ ]:
"""
postoje invoice price 0 narudzbe, onda gm% bude nan i nemogu korelaciju gledati
ponekad je inovice price 0, a product cost nije, mogao bih onda gm% postaviti na -max
order qty i deliver qty se razlikuju
invoice dateovi neki su invalidni


nisam zadovoljan sve skupa jer postoje klasteri s po jednom osobom, možda su to ovi izdvojeni iz skupine na slici
"""

In [ ]:
# DBSCAN with artificial well-separated data
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_blobs

# Generate artificial well-separated data
n_samples = 300
centers = [[0, 0], [5, 5], [10, 0], [5, -5]]
X_artificial, y_true = make_blobs(
    n_samples=n_samples,
    centers=centers,
    cluster_std=0.4,
    random_state=42
)

# Apply DBSCAN
dbscan_artificial = DBSCAN(eps=1.0, min_samples=5)
labels_artificial = dbscan_artificial.fit_predict(X_artificial)

n_clusters_artificial = len(set(labels_artificial)) - (1 if -1 in labels_artificial else 0)
n_noise_artificial = list(labels_artificial).count(-1)

print(f"Number of clusters: {n_clusters_artificial}")
print(f"Number of noise points: {n_noise_artificial}")

# Visualize
plt.figure(figsize=(10, 8))
unique_labels = set(labels_artificial)
colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))

for k, col in zip(unique_labels, colors):
    if k == -1:
        col = [0, 0, 0, 1]  # Black for noise
        marker = 'x'
    else:
        marker = 'o'
    
    class_member_mask = (labels_artificial == k)
    xy = X_artificial[class_member_mask]
    plt.scatter(xy[:, 0], xy[:, 1], c=[col], marker=marker, s=100, 
                edgecolors='k', label=f'Cluster {k}' if k != -1 else 'Noise')

plt.title('DBSCAN Clustering on Artificial Data')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# More challenging artificial data with varying densities and cluster characteristics
from sklearn.datasets import make_moons, make_circles
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# Cluster 1: Dense spherical cluster
cluster1 = np.random.normal(loc=[2, 2], scale=0.3, size=(200, 2))

# Cluster 2: Sparse elongated cluster (requires different eps)
cluster2 = np.random.normal(loc=[8, 8], scale=1.2, size=(100, 2))

# Cluster 3: Dense non-spherical (crescent)
crescent, _ = make_moons(n_samples=150, noise=0.08)
crescent = crescent * 1.5 + [5, 2]

# Cluster 4: Circular cluster with hole (ring)
angles = np.random.uniform(0, 2*np.pi, 150)
radii = np.random.normal(3, 0.3, 150)
cluster4 = np.column_stack([5 + radii*np.cos(angles), 12 + radii*np.sin(angles)])

# Add noise points throughout the space
noise = np.random.uniform(-2, 15, size=(50, 2))

# Combine all data
X_complex = np.vstack([cluster1, cluster2, crescent, cluster4, noise])
np.random.shuffle(X_complex)

# Visualize the raw complex data
plt.figure(figsize=(12, 8))
plt.scatter(X_complex[:, 0], X_complex[:, 1], alpha=0.6, s=50)
plt.title('Challenging Artificial Data:\nVarying Densities, Non-Spherical Shapes, Different Cluster Sizes')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Total data points: {len(X_complex)}")
print("Data characteristics:")
print("  - Cluster 1: Dense spherical (200 points, std=0.3)")
print("  - Cluster 2: Sparse spherical (100 points, std=1.2)")
print("  - Cluster 3: Dense crescent shape (150 points)")
print("  - Cluster 4: Ring pattern (150 points)")
print("  - Noise: Random points (50 points)")
print("→ DBSCAN parameter choice will significantly affect results!")

In [ ]:
# Hyperparameter search for DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Define parameter ranges
eps_values = np.arange(0.1, 1.0, 0.1)
min_samples_values = [3, 5, 10, 15, 20]

# Store results
results = []

for eps in eps_values:
    for min_samples in min_samples_values:
        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
        labels = dbscan.fit_predict(X_complex)
        
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = list(labels).count(-1)
        
        # Calculate metrics only if we have valid clusters (more than 1 cluster and not all noise)
        if n_clusters > 0 and n_noise < len(X_complex) - 1:
            silhouette = silhouette_score(X_complex, labels)
            davies_bouldin = davies_bouldin_score(X_complex, labels)
        else:
            silhouette = -1  # Invalid
            davies_bouldin = float('inf')  # Invalid
        
        results.append({
            'eps': eps,
            'min_samples': min_samples,
            'n_clusters': n_clusters,
            'n_noise': n_noise,
            'silhouette': silhouette,
            'davies_bouldin': davies_bouldin
        })

# Convert to DataFrame for easier analysis
results_df = pd.DataFrame(results)

# Filter valid results (silhouette > -1)
valid_results = results_df[results_df['silhouette'] > -1].copy()

if len(valid_results) > 0:
    # Find best parameters by silhouette score
    best_silhouette = valid_results.loc[valid_results['silhouette'].idxmax()]
    print("=" * 60)
    print("BEST PARAMETERS BY SILHOUETTE SCORE:")
    print("=" * 60)
    print(f"eps: {best_silhouette['eps']:.2f}")
    print(f"min_samples: {int(best_silhouette['min_samples'])}")
    print(f"Number of clusters: {int(best_silhouette['n_clusters'])}")
    print(f"Noise points: {int(best_silhouette['n_noise'])}")
    print(f"Silhouette score: {best_silhouette['silhouette']:.4f}")
    print(f"Davies-Bouldin Index: {best_silhouette['davies_bouldin']:.4f}")
    print()
    
    # Show top 5 parameter combinations
    print("=" * 60)
    print("TOP 5 PARAMETER COMBINATIONS (by Silhouette Score):")
    print("=" * 60)
    print(valid_results.nlargest(5, 'silhouette')[['eps', 'min_samples', 'n_clusters', 'silhouette', 'davies_bouldin']])
else:
    print("No valid clustering found. Try adjusting the parameter ranges.")

In [ ]:
# Visualize clustering results for different parameter combinations
if len(valid_results) > 0:
    # Get top 6 parameter combinations
    top_params = valid_results.nlargest(6, 'silhouette')
    
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.flatten()
    
    for idx, (_, row) in enumerate(top_params.iterrows()):
        eps = row['eps']
        min_samples = int(row['min_samples'])
        silhouette = row['silhouette']
        n_clusters = int(row['n_clusters'])
        
        # Re-run DBSCAN with these parameters
        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
        labels = dbscan.fit_predict(X_complex)
        
        # Plot
        ax = axes[idx]
        unique_labels = set(labels)
        colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))
        
        for k, col in zip(unique_labels, colors):
            if k == -1:
                col = [0, 0, 0, 1]
                marker = 'x'
            else:
                marker = 'o'
            
            class_member_mask = (labels == k)
            xy = X_complex[class_member_mask]
            ax.scatter(xy[:, 0], xy[:, 1], c=[col], marker=marker, s=50, 
                      edgecolors='k', alpha=0.7)
        
        ax.set_title(f'eps={eps:.2f}, min_samples={min_samples}\nSilhouette={silhouette:.4f} ({n_clusters} clusters)', 
                    fontsize=10, fontweight='bold')
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Tie-breaking analysis for similar parameter combinations
if len(valid_results) > 0:
    # Get top candidates within a small margin
    best_score = valid_results['silhouette'].max()
    margin = 0.05  # 5% margin
    candidates = valid_results[valid_results['silhouette'] >= best_score - margin].copy()
    candidates = candidates.sort_values('silhouette', ascending=False)
    

    print("=" * 80)
    print("TIE-BREAKING ANALYSIS FOR SIMILAR PARAMETERS")
    print("=" * 80)
    print(f"\nCandidates within {margin*100:.1f}% of best score:\n")
    print(candidates[['eps', 'min_samples', 'n_clusters', 'n_noise', 'silhouette', 'davies_bouldin']])
    
    print("\n" + "=" * 80)
    print("DECISION CRITERIA:")
    print("=" * 80)
    
    for idx, (_, row) in enumerate(candidates.head(3).iterrows(), 1):
        eps = row['eps']
        min_samples = int(row['min_samples'])
        
        print(f"\nOption {idx}: eps={eps:.2f}, min_samples={min_samples}")
        print(f"  - Silhouette: {row['silhouette']:.4f}")
        print(f"  - Davies-Bouldin: {row['davies_bouldin']:.4f}")
        print(f"  - Clusters: {int(row['n_clusters'])}, Noise: {int(row['n_noise'])}")
        
        # Calculate sensitivity: check nearby parameter combinations
        nearby = valid_results[
            ((valid_results['eps'] - eps).abs() <= 0.1) & 
            ((valid_results['min_samples'] - min_samples).abs() <= 5) &
            ~((valid_results['eps'] == eps) & (valid_results['min_samples'] == min_samples))
        ]
        
        if len(nearby) > 0:
            avg_nearby_score = nearby['silhouette'].mean()
            score_drop = row['silhouette'] - avg_nearby_score
            print(f"  - Robustness: Average nearby score = {avg_nearby_score:.4f} (drop: {score_drop:.4f})")
            if score_drop > 0.1:
                print(f"    ✓ ROBUST - significant performance drop with nearby params")
            else:
                print(f"    ~ SENSITIVE - performance similar to nearby params")
        
        # Simplicity score (prefer smaller parameters)
        simplicity = (1.0 / eps) + (1.0 / min_samples)
        print(f"  - Simplicity score: {simplicity:.2f} (higher = simpler params)")
        
        # Balance score (prefer balanced cluster/noise ratio)
        cluster_ratio = int(row['n_clusters']) / (len(X_complex) - int(row['n_noise']))
        print(f"  - Cluster balance: {int(row['n_clusters'])} clusters, {cluster_ratio:.2f} avg points per cluster")
    
    print("\n" + "=" * 80)
    print("RECOMMENDATION:")
    print("=" * 80)
    best_choice = candidates.iloc[0]
    print(f"Choose: eps={best_choice['eps']:.2f}, min_samples={int(best_choice['min_samples'])}")
    print("This combination offers the best overall balance of:")
    print("  - Highest Silhouette score (primary metric)")
    print("  - Good Davies-Bouldin Index")
    print("  - Reasonable number of clusters")
    print("=" * 80)